# RAG Evaluation Workshop

Learn how to evaluate RAG systems using various metrics and frameworks.

## Setup

In [ ]:
!pip install ragas datasets langchain-openai

## Prepare Evaluation Data

In [1]:
from datasets import Dataset

# Sample evaluation data
eval_data = {
    "question": [
        "What is RAG?",
        "How does vector search work?",
        "What are the benefits of RAG?",
        "What is the difference between RAG and fine-tuning?",
        "When should you use RAG?"
    ],
    "answer": [
        "RAG stands for Retrieval-Augmented Generation...",
        "Vector search uses embeddings to find similar...",
        "RAG helps reduce hallucinations and provides...",
        "Fine-tuning adapts a model to specific tasks...",
        "You should use RAG when you need to query..."
    ],
    "contexts": [
        ["RAG is a framework...", "It combines retrieval..."],
        ["Vector embeddings...", "Similarity search..."],
        ["Benefits include...", "Reduced hallucinations..."],
        ["Fine-tuning is...", "RAG is..."],
        ["Use RAG for...", "External knowledge..."]
    ],
    "ground_truth": [
        "RAG is Retrieval-Augmented Generation, a technique...",
        "Vector search works by converting text to embeddings...",
        "RAG provides fresh knowledge, reduces hallucinations...",
        "Fine-tuning modifies model weights, RAG retrieves...",
        "Use RAG for question answering, knowledge bases..."
    ]
}

dataset = Dataset.from_dict(eval_data)
print(dataset)

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})


## Evaluate with RAGAS

In [5]:
from ragas import evaluate

# Import metrics as classes (not modules)
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

# Initialize the metrics
faithfulness_score = Faithfulness()
answer_relevancy_score = AnswerRelevancy()
context_precision_score = ContextPrecision()
context_recall_score = ContextRecall()

# Run evaluation (requires OPENAI_API_KEY set)
# results = evaluate(
#     dataset=dataset,
#     metrics=[
#         faithfulness_score,
#         answer_relevancy_score,
#         context_precision_score,
#         context_recall_score
#     ]
# )
# print(results)

print("RAGAS metrics initialized successfully!")
print("Note: To run evaluation, set OPENAI_API_KEY environment variable.")

RAGAS metrics initialized successfully!
Note: To run evaluation, set OPENAI_API_KEY environment variable.


/var/folders/1d/rhn4tf2x0fdd1yv24ptzskxm0000gp/T/ipykernel_46352/3675029564.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/var/folders/1d/rhn4tf2x0fdd1yv24ptzskxm0000gp/T/ipykernel_46352/3675029564.py:4: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/var/folders/1d/rhn4tf2x0fdd1yv24ptzskxm0000gp/T/ipykernel_46352/3675029564.py:4: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics

## Custom LLM-as-Judge Evaluator

In [6]:
from langchain_ollama import ChatOllama

class RAGEvaluator:
    """Evaluate RAG using LLM as judge."""
    
    def __init__(self, llm=None):
        self.llm = llm or ChatOllama(model="llama3.2")
    
    def evaluate_faithfulness(self, question, answer, context):
        prompt = f"""Rate faithfulness 0-10:\n\nQuestion: {question}\n\nContext: {context}\n\nAnswer: {answer}\n\nOnly respond with a number:"""
        return float(self.llm.invoke(prompt).content.strip()) / 10
    
    def evaluate_answer_quality(self, question, answer):
        prompt = f"""Rate answer quality 0-10:\n\nQuestion: {question}\n\nAnswer: {answer}\n\nOnly respond with a number:"""
        return float(self.llm.invoke(prompt).content.strip()) / 10
    
    def full_evaluation(self, question, answer, contexts):
        context_str = "\n\n".join(contexts)
        return {
            "faithfulness": self.evaluate_faithfulness(question, answer, context_str),
            "answer_quality": self.evaluate_answer_quality(question, answer)
        }

evaluator = RAGEvaluator()

# Evaluate all questions
for i in range(len(eval_data["question"])):
    result = evaluator.full_evaluation(
        eval_data["question"][i],
        eval_data["answer"][i],
        eval_data["contexts"][i]
    )
    print(f"Q{i+1}: Faithfulness={result['faithfulness']:.2f}, Quality={result['answer_quality']:.2f}")

Q1: Faithfulness=0.70, Quality=0.60
Q2: Faithfulness=0.60, Quality=0.70
Q3: Faithfulness=0.80, Quality=0.60
Q4: Faithfulness=0.80, Quality=0.70
Q5: Faithfulness=0.80, Quality=0.80


## Retrieval Metrics

In [7]:
def precision_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & set(relevant)) / len(relevant)

# Example
retrieved = ["doc1", "doc2", "doc3", "doc4", "doc5"]
relevant = ["doc1", "doc3", "doc5"]

print(f"Precision@3: {precision_at_k(retrieved, relevant, 3):.2f}")
print(f"Recall@3: {recall_at_k(retrieved, relevant, 3):.2f}")
print(f"Precision@5: {precision_at_k(retrieved, relevant, 5):.2f}")
print(f"Recall@5: {recall_at_k(retrieved, relevant, 5):.2f}")

Precision@3: 0.67
Recall@3: 0.67
Precision@5: 0.60
Recall@5: 1.00


## Benchmarking Function

In [8]:
import time
from langchain_core.runnables import RunnableLambda

def benchmark_rag(rag_pipeline, test_queries):
    """Benchmark a RAG pipeline."""
    results = []
    
    for query in test_queries:
        start = time.time()
        response = rag_pipeline.invoke({"query": query})
        latency = (time.time() - start) * 1000
        
        # Handle different response types
        if isinstance(response, dict):
            answer = response.get("result", str(response))
        else:
            answer = str(response)
        
        results.append({
            "query": query,
            "latency_ms": latency,
            "answer": answer[:100]
        })
    
    avg_latency = sum(r["latency_ms"] for r in results) / len(results)
    
    return {
        "results": results,
        "avg_latency_ms": avg_latency,
        "total_queries": len(results)
    }

# ===== WORKING EXAMPLE =====

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Sample documents
documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a generative AI model.",
    "RAG helps reduce hallucinations by grounding responses in retrieved context.",
    "Benefits of RAG include: fresh knowledge, reduced hallucinations, traceability, and cost-effective updates.",
    "Vector search works by converting text into embeddings and finding similar vectors.",
    "Embeddings are numerical representations of text that capture semantic meaning."
]

# Create embeddings and vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
docs = text_splitter.create_documents(documents)
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)
retriever = vectorstore.as_retriever()

# Create LLM
llm = ChatOllama(model="llama3.2")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def extract_question(inputs):
    """Extract question from dict or string."""
    if isinstance(inputs, dict):
        return inputs.get("query", inputs.get("question", ""))
    return inputs

def create_prompt(context, question):
    return f"""Based on the following context, answer the question.

Context: {context}

Question: {question}

Answer:"""

# Fixed RAG chain - extracts question, retrieves, then generates
simple_rag_chain = (
    RunnableLambda(extract_question)
    | retriever
    | format_docs
    | (lambda ctx: {"context": ctx, "question": extract_question({"query": ""})})
    | RunnableLambda(lambda x: llm.invoke(create_prompt(x["context"], x["question"])))
)

# Run benchmark
test_queries = [
    "What is RAG?",
    "How does it work?",
    "Benefits?"
]

print("Running benchmark...")
benchmark = benchmark_rag(simple_rag_chain, test_queries)
print(f"Average latency: {benchmark['avg_latency_ms']:.2f}ms")
print(f"Total queries: {benchmark['total_queries']}")
for r in benchmark['results']:
    print(f"  - {r['query']}: {r['latency_ms']:.2f}ms")

Running benchmark...
Average latency: 350.27ms
Total queries: 3
  - What is RAG?: 417.54ms
  - How does it work?: 160.59ms
  - Benefits?: 472.68ms


## Next Steps

- Add more test questions
- Compare different RAG configurations
- Set up continuous evaluation
- Create evaluation dashboards